# Lesson 23: Model Fitting, Robust Estimation, and RANSAC

Lesson 22 fit a homography from point correspondences, and Lesson 19's feature matching is exactly where those correspondences come from &mdash; but feature matching is never perfect: some matches are simply wrong. This lesson confronts that directly: ordinary least-squares fitting is catastrophically fragile to even a few bad points, and **RANSAC** is the standard fix, quietly running underneath the `cv2.RANSAC` flag used (without explanation) in Lesson 22 &mdash; and again in the next lesson's essential-matrix estimation.

In [ ]:
import numpy as np
import cv2
import matplotlib.pyplot as plt

## Why least squares breaks under outliers

Least-squares fitting minimizes the *sum of squared* residuals. Squaring means a single point far from the model contributes an enormous amount to the total error, which drags the whole fit toward accommodating it &mdash; even at the cost of moving away from all the other, correct points.

In [ ]:
rng = np.random.default_rng(0)
true_slope, true_intercept = 2.0, 5.0

x_inliers = rng.uniform(0, 10, 40)
y_inliers = true_slope * x_inliers + true_intercept + rng.normal(0, 0.5, 40)

x_outliers = rng.uniform(0, 10, 15)
y_outliers = rng.uniform(-20, 40, 15)  # unrelated to the line at all

x = np.concatenate([x_inliers, x_outliers])
y = np.concatenate([y_inliers, y_outliers])

A = np.vstack([x, np.ones_like(x)]).T
slope_ols, intercept_ols = np.linalg.lstsq(A, y, rcond=None)[0]

print(f'true line:                 y = {true_slope:.2f}x + {true_intercept:.2f}')
print(f'ordinary least squares:    y = {slope_ols:.2f}x + {intercept_ols:.2f}   <- dragged off by outliers')

xs = np.array([0, 10])
plt.scatter(x_inliers, y_inliers, c='tab:blue', label='inliers')
plt.scatter(x_outliers, y_outliers, c='tab:red', marker='x', label='outliers')
plt.plot(xs, true_slope * xs + true_intercept, '--', color='gray', label='true line')
plt.plot(xs, slope_ols * xs + intercept_ols, color='tab:orange', label='OLS fit')
plt.legend(fontsize=8)
plt.title('Ordinary least squares, corrupted by 15 outliers among 40 inliers')
plt.show()

## RANSAC: fit from the inside out

**RANSAC** (RANdom SAmple Consensus, <a href="../references.html#fischler-bolles-1981">Fischler & Bolles, 1981</a><span class="landmark-paper">&#9733;</span>) flips the strategy: instead of using *all* the data and hoping outliers don't matter, it repeatedly picks the *smallest possible* random subset needed to define a candidate model, counts how many of the remaining points agree with it (the **consensus set**), and keeps whichever candidate has the most agreement. A final least-squares refit on just that winning inlier set gives the polished result &mdash; least squares is fine once outliers are already excluded, which is exactly what makes this two-stage combination work.

In [ ]:
def ransac_line(x, y, threshold=2.0, n_iterations=200, rng=None):
    rng = rng or np.random.default_rng()
    best_inliers, best_count = None, -1

    for _ in range(n_iterations):
        i, j = rng.choice(len(x), 2, replace=False)     # minimal sample: 2 points define a line
        if x[i] == x[j]:
            continue
        m = (y[j] - y[i]) / (x[j] - x[i])
        b = y[i] - m * x[i]
        distance = np.abs(m * x - y + b) / np.sqrt(m**2 + 1)
        inliers = distance < threshold
        if inliers.sum() > best_count:
            best_count, best_inliers = inliers.sum(), inliers

    A = np.vstack([x[best_inliers], np.ones(best_inliers.sum())]).T
    slope, intercept = np.linalg.lstsq(A, y[best_inliers], rcond=None)[0]  # final refit, inliers only
    return slope, intercept, best_inliers

slope_ransac, intercept_ransac, inlier_mask = ransac_line(x, y, rng=np.random.default_rng(1))

print(f'true line:   y = {true_slope:.2f}x + {true_intercept:.2f}')
print(f'RANSAC fit:  y = {slope_ransac:.2f}x + {intercept_ransac:.2f}')
print(f'inliers found: {inlier_mask.sum()} / {len(x)}  (planted {40} true inliers)')

plt.scatter(x[inlier_mask], y[inlier_mask], c='tab:blue', label='found inliers')
plt.scatter(x[~inlier_mask], y[~inlier_mask], c='tab:red', marker='x', label='rejected as outliers')
plt.plot(xs, slope_ransac * xs + intercept_ransac, color='tab:green', label='RANSAC fit')
plt.legend(fontsize=8)
plt.title('RANSAC recovers the true line despite 27% outlier contamination')
plt.show()

## How many iterations does RANSAC need?

If a fraction $w$ of the data are inliers, and the model needs a minimal sample of $n$ points, the probability that any one random sample is entirely inliers is $w^n$. To be at least `p` confident of drawing an all-inlier sample at least once across $N$ independent tries:

$$N = \frac{\log(1-p)}{\log(1-w^n)}$$

Fewer inliers, or a larger minimal sample size, both blow this up fast.

In [ ]:
def required_iterations(inlier_fraction, sample_size, confidence=0.99):
    return np.log(1 - confidence) / np.log(1 - inlier_fraction**sample_size)

print(f'{"inlier %":>10} {"line (n=2)":>12} {"homography (n=4)":>18}')
for w in [0.9, 0.7, 0.5, 0.3]:
    n_line = int(np.ceil(required_iterations(w, 2)))
    n_homog = int(np.ceil(required_iterations(w, 4)))
    print(f'{100*w:>9.0f}% {n_line:>12} {n_homog:>18}')

Fitting a line only ever needs 2 points, so even fairly heavy contamination (50% outliers) needs just a few dozen iterations. Fitting a homography needs a minimal sample of 4 points (Lesson 22), so the same inlier fraction needs an order of magnitude more iterations &mdash; the price of a more complex model.

## RANSAC in practice: homography estimation with bad matches

This is exactly the scenario from Lesson 22's `cv2.findHomography`, now with deliberately injected garbage correspondences &mdash; standing in for the mismatched features that real feature matching (Lesson 19) inevitably produces even after a ratio test.

In [ ]:
H_true = np.array([[1, 0.2, 10], [0.05, 1, 5], [0.0008, 0.0003, 1]])

def apply_homography(points, H):
    homogeneous = np.hstack([points, np.ones((len(points), 1))])
    transformed = (H @ homogeneous.T).T
    return transformed[:, :2] / transformed[:, 2:3]

pts1_correct = rng.uniform(0, 200, (30, 2))
pts2_correct = apply_homography(pts1_correct, H_true)

pts1_bad = rng.uniform(0, 200, (15, 2))
pts2_bad = rng.uniform(0, 300, (15, 2))  # unrelated -- simulated bad matches

pts1 = np.vstack([pts1_correct, pts1_bad])
pts2 = np.vstack([pts2_correct, pts2_bad])

H_ransac, mask = cv2.findHomography(pts1, pts2, cv2.RANSAC, 3.0)
H_plain, _ = cv2.findHomography(pts1, pts2, 0)  # method=0: plain least squares, no outlier rejection

def mean_reprojection_error(H, pts1, pts2):
    return np.linalg.norm(apply_homography(pts1, H) - pts2, axis=1).mean()

print(f'inliers found by RANSAC: {int(mask.sum())} / {len(mask)}  (30 correspondences were genuinely correct)')
print()
print(f'mean reprojection error on the TRUE inliers:')
print(f'  RANSAC fit: {mean_reprojection_error(H_ransac, pts1_correct, pts2_correct):.2e} pixels')
print(f'  plain fit:  {mean_reprojection_error(H_plain, pts1_correct, pts2_correct):.2e} pixels')

RANSAC finds exactly the 30 genuine correspondences and reconstructs the homography to essentially machine precision. The plain least-squares fit, given the exact same data, is off by many orders of magnitude more error &mdash; effectively useless for anything requiring pixel-level accuracy &mdash; because 15 bad matches out of 45 (33%) was more than enough to noticeably corrupt an unweighted sum-of-squares fit. The `cv2.findHomography(..., cv2.RANSAC, ...)` call back in Lesson 22 was quietly doing exactly this, and the next lesson's `cv2.findEssentialMat(..., cv2.RANSAC)` call will do the same.

## A gentler alternative: robust loss functions

RANSAC makes a hard inlier/outlier decision. An alternative family, **M-estimators**, instead reweights every point's contribution smoothly &mdash; e.g. the **Huber loss** behaves like ordinary squared error for small residuals but switches to linear (much less aggressive) growth beyond a threshold, so a single very-wrong point can no longer dominate the total cost the way it does in `Loss = residual^2`. RANSAC and M-estimators are complementary in practice: RANSAC is excellent at rejecting *gross* outliers (completely wrong matches), while an M-estimator refinement afterward can down-weight smaller, more subtle deviations among the remaining inliers.

### Exercise

1. Increase `ransac_line`'s outlier count until roughly 70% of the points are outliers. Does 200 iterations remain enough to reliably recover the true line? Use the `required_iterations` formula to check whether 200 is even theoretically sufficient at that contamination level.
2. Lower the RANSAC distance `threshold` for the homography example from `3.0` to `0.5`. Does the number of found inliers change, and why might too-strict a threshold actually start rejecting *genuine* inliers (hint: think about what noise, not outliers, does to correct correspondences)?
3. Implement a simple Huber-loss reweighted least squares for the line-fitting example (iteratively: fit, compute residuals, downweight points with `|residual| > delta` by `delta/|residual|`, refit) and compare its result to RANSAC's on the same contaminated data.